In [1]:
import numpy as np

rng = np.random.default_rng(7)

class BENCHMARK():
    def __init__(self,n_inputs,hidden=5):
        self.W1 = rng.normal(0,1,size=(n_inputs,hidden)) * 0.5
        self.b1 = np.zeros(hidden)
        self.w2 = rng.normal(0,1,size=hidden) * 0.5
        self.b2 = 0.0
        self.mean = []
        
    def forward(self,x):
        self.h = np.tanh(x @ self.W1 + self.b1)
        s = self.h @ self.w2 + self.b2
        return np.tanh(s) 
    
    def learn(self,x,t,lr):
        y = self.forward(x) 
        e = t - y
        blame = e * (1 - y ** 2)
        h_blame = blame * self.w2 * (1 - self.h ** 2)
        self.W1 += lr * np.outer(x,h_blame)
        self.b1 += lr * h_blame
        self.w2 += lr * blame * self.h
        self.b2 += lr * blame
        self.mean.append(abs(e))
        
    def print_weights(self):
        print(self.W1)
        print(self.b1)
        print(self.w2)
        print(self.b2)
        
    def reset_memory(self):
        self.mean = []
        
n = BENCHMARK(3,2)
x = np.array([3,5,2])
true_w = np.array([0.5,-0.3,0.6])
print(n.forward(x))

-0.5975567048214637


In [2]:
wave = 0.5 * np.sin(np.linspace(0,20,500)) 
X = []
Y = []
for i in range(len(wave) - 3):
    X.append(wave[i:i + 3])
    Y.append(wave[i + 3])

X = np.array(X)
Y = np.array(Y)

wave2 = 0.5 * np.sin(8 * np.linspace(0,20,500)) 
X2 = []
Y2 = []
for i in range(len(wave2) - 3):
    X2.append(wave2[i:i + 3])
    Y2.append(wave2[i + 3])

X2 = np.array(X2)
Y2 = np.array(Y2)

xa = rng.uniform(0.1,0.5,500)
xb = rng.uniform(-0.5,-0.1,500)
XA = xa[:,None]
XB = xb[:,None]
YA = np.full(500,-0.2)
YB = np.full(500,-0.5)

print(X.shape,Y.shape)
print(X2.shape,Y2.shape)
print(XA.shape,YA.shape)
print(XB.shape,YB.shape)

(497, 3) (497,)
(497, 3) (497,)
(500, 1) (500,)
(500, 1) (500,)


In [3]:
def memory_base(seed,hidden=4):
    global rng
    rng = np.random.default_rng(seed)
    n = BENCHMARK(1,hidden)
    for epoch in range(30):
        for i in range(400):
            n.learn(XA[i],YA[i],0.1)
    n.reset_memory()
    for i in range(400,500):
        n.learn(XA[i],YA[i],0.0)
    base_a = sum(n.mean) / len(n.mean)

    for epoch in range(30):
        for i in range(400):
            n.learn(XB[i],YB[i],0.1)
    n.reset_memory()
    for i in range(400,500):
        n.learn(XB[i],YB[i],0.0)
    base_b = sum(n.mean) / len(n.mean)

    n.reset_memory()
    for i in range(400,500):
        n.learn(XA[i],YA[i],0.0)
    base_aa = sum(n.mean) / len(n.mean)
    return base_a,base_b,base_aa

In [4]:
threshold = 0.05

class STO_tree():
    def __init__(self,grow=True,gate=None):
        self.branches = {} 
        self.b = 0.0
        self.next_id = 0
        self.exits = []
        self.rval = 0
        self.activation = 0.0
        self.mean = [] 
        self.gain = 1.0
        self.act_avg = 0.5
        self.cool = 0
        self.grow = grow
        self.gate_val = gate
        self.s = 1.0
        self.p_avg = 0.0
        self.steps = 0
        self.err_avg = 0.5
        self.lock_at = None
        
    def roots(self,n_inputs):
        self.n_roots = n_inputs
        for i in range(n_inputs):
            self.exits.append(self.next_id)
            self.branches[self.next_id] = {"exit_type": 1,
            "parent": None,
            "children": [],
            "weight": 1 + rng.normal(0,0.1),
            "pos": "for later",
            "entry": 0.0,
            "exit": 0.0,
            "gain": 0.0,
            "stiff": 0.0,
            "activity": 0.5,
            "avg_blame": 0.0,
            "input_idx": i,
            "involvment": 1.0}
            self.next_id += 1
            
    def branch(self,parent,am,inputs = None):
        self.branches[parent]["exit_type"] = 0
        self.exits.remove(parent)
        for i in range(am):
            self.exits.append(self.next_id)
            self.branches[self.next_id] = {"exit_type": 1,
            "parent": parent,
            "children": [],
            "weight": 1 + rng.normal(0,0.1),
            "pos": "for later",
            "entry": 0.0,
            "exit": 0.0,
            "gain": 0.0,
            "stiff": 0.0,
            "activity": 0.5,
            "avg_blame": 0.0,
            "input_idx": (inputs[i] if inputs else self.branches[parent]["input_idx"]),
            "involvment": self.branches[parent]["involvment"] / am}
            self.branches[parent]["children"].append(self.next_id)
            self.next_id += 1
            
    def forward(self,x):
        self.rval = 0
        for a,b in enumerate(self.exits): #inputs
            self.branches[b]["entry"] = x[self.branches[b]["input_idx"]]
            self.branches[b]["exit"] = np.tanh(self.branches[b]["weight"] * self.branches[b]["entry"])
            if abs(self.branches[b]["entry"]) > threshold:
                self.branches[b]["gain"] = self.branches[b]["exit"] / self.branches[b]["entry"]
            else:
                self.branches[b]["gain"] = 1.0
                
        for be,bid in reversed([(e,self.branches[e]) for e in self.branches if self.branches[e]["exit_type"] == 0]): #paren
            val = 0
            for c in bid["children"]:
                val += self.branches[c]["exit"]
            self.branches[be]["entry"] = val / len(bid["children"])
            u = self.branches[be]["weight"] * self.branches[be]["entry"]
            self.branches[be]["exit"] = u * np.exp(0.5 - 0.5 * u ** 2)
            
            if abs(self.branches[be]["weight"]) <0.001:
                self.branches[be]["gain"] = 0
            else:
                center_entry = 1 / self.branches[be]["weight"]
                d = self.branches[be]["entry"] - center_entry
                if abs(d) > threshold:
                    self.branches[be]["gain"] = (self.branches[be]["exit"] - 1) / d
                else:
                    self.branches[be]["gain"] = 0
        for r in range(self.n_roots):
            self.rval += self.branches[r]["exit"]
        self.activation = self.rval + self.b
        y = self.s * np.tanh(self.activation)
        if abs(self.activation) > threshold:
            self.gain = y / self.activation
        else:
            self.gain = self.s
        return y
    
    def learn(self,x,t,lr):
        y = self.forward(x) 
        self.mean.append(abs(t - y))
        if self.act_avg < 0.1:
            self.act_avg = 0.5
        self.act_avg = 0.99 * self.act_avg + 0.01 * abs(self.rval)
        self.p_avg = 0.99 * self.p_avg + 0.01 * t * self.rval
        blame = (t - y) / self.gain
        share = blame / (self.n_roots + 1)
        to_sprout = []
        reach = {} 
        for ro in range(self.n_roots):
            reach[ro] = 1.0
        for o in self.branches:
            if self.branches[o]["parent"] is not None:
                reach[o] = reach[self.branches[o]["parent"]] * self.branches[self.branches[o]["parent"]]["gain"]
            its_blame = share * reach[o] * self.branches[o]["involvment"]
            if abs(self.branches[o]["entry"]) > threshold and (self.gate_val is None or abs(self.branches[o]["exit"]) > self.gate_val * self.branches[o]["activity"]):
                change_weight = lr * its_blame * np.sign(self.branches[o]["entry"])
                if self.branches[o]["children"] != []:
                    change_weight *= (1 if abs(self.branches[o]["weight"] * self.branches[o]["entry"]) < 1 else (-1))
                self.branches[o]["weight"] += change_weight
            self.branches[o]["activity"] = 0.99 * self.branches[o]["activity"] + 0.01 * abs(self.branches[o]["exit"])
            self.branches[o]["avg_blame"] = 0.99 * self.branches[o]["avg_blame"] + 0.01 * abs(its_blame)
            if self.branches[o]["children"] == [] and self.branches[o]["avg_blame"] > 0.015 and self.grow:
                to_sprout.append(o)
            if self.act_avg < 0.1:
                self.branches[o]["weight"] = 1 + rng.normal(0,0.1)
                self.branches[o]["activity"] = 0.5
                self.branches[o]["avg_blame"] = 0
        self.b += lr * share
        self.err_avg = 0.99 *self.err_avg + 0.01 * abs(t - y)
        if self.lock_at is None and (self.err_avg < 0.15 or self.steps >= 8000):
            self.lock_at = self.steps + 300
        if self.lock_at is None or self.steps < self.lock_at: 
            if self.p_avg >= 0:
                self.s = 1.0
            else:
                self.s = -1.0
        self.steps += 1
        self.cool -= 1
        for sprout_me in to_sprout:
            if len(self.branches) < 15 and self.cool <= 0 and max(self.exits,key=lambda i: self.branches[i]["avg_blame"]) == sprout_me:
                self.branch(sprout_me,2)
                self.cool = 395
        
    def reset_memory(self):
        self.mean = []           
            
            
#hope this cutie works, so lovelyyyyyyyy
tree = STO_tree()
tree.roots(2)
tree.branch(0,2)
print(tree.forward(np.array([0.5,0.4,0.3])))
for repeat in range(20):
    tree.learn(np.array([0.5, 0.4, 0.3]), 0.9, 0.1)
print(tree.forward(np.array([0.5,0.4,0.3])))
print()
for test in tree.branches:
    print("-",test)
    print(tree.branches[test]["entry"])
    print(tree.branches[test]["exit"])
    print()
print(tree.branches[3])

0.7985739068114384
0.8448232890376319

- 0
0.48351360858411874
0.7424016977578245

- 1
0.4
0.4233439263529066

- 2
0.5
0.510309195763826

- 3
0.5
0.4567180214044114

{'exit_type': 1, 'parent': 0, 'children': [], 'weight': np.float64(0.9863127710254195), 'pos': 'for later', 'entry': np.float64(0.5), 'exit': np.float64(0.4567180214044114), 'gain': np.float64(0.9134360428088228), 'stiff': 0.0, 'activity': np.float64(0.4914356166858746), 'avg_blame': np.float64(0.0018452241347286286), 'input_idx': 0, 'involvment': 0.5}


In [5]:
def memory_test(seed,grow,gate):
    global rng
    rng = np.random.default_rng(seed)
    test_me = STO_tree(grow=grow,gate=gate)
    test_me.roots(1)
    for epoch in range(30):
        for i in range(400):
            test_me.learn(XA[i],YA[i],0.1)
    test_me.reset_memory()
    for i in range(400,500):
        test_me.learn(XA[i],YA[i],0.0)
    test_a = sum(test_me.mean) / len(test_me.mean)

    for epoch in range(30):
        for i in range(400):
            test_me.learn(XB[i],YB[i],0.1)
            
    test_me.reset_memory()
    for i in range(400,500):
        test_me.learn(XB[i],YB[i],0.0)
    test_b = sum(test_me.mean) / len(test_me.mean)

    test_me.reset_memory()
    for i in range(400,500):
        test_me.learn(XA[i],YA[i],0.0)
    test_aa = sum(test_me.mean) / len(test_me.mean)
    return test_a,test_b,test_aa

In [6]:

def output_correct(forward_result):
    correct_value = 0
    if forward_result[0] < 0:
        correct_value += 1
    if forward_result[1] > 0:
        correct_value += 1 
    if forward_result[2] > 0:
        correct_value += 1
    if forward_result[3] < 0:
        correct_value += 1
    return correct_value

def test_seed(seed):
    global rng
    rng = np.random.default_rng(seed)
    r1,r1_2,r2,r2_2 = 0,0,0,0
    P = np.array([(0,0),(0,1),(1,0),(1,1)])
    T = np.array([-0.5,0.5,0.5,-0.5])
    xor_test = STO_tree(grow=True)
    xor_test.roots(1)
    xor_test.branch(0,2,[0,1])
    xor_test_2 = STO_tree(grow=False)
    xor_test_2.roots(1)
    xor_test_2.branch(0,2,[0,1])
    xor_base = BENCHMARK(2,4)
    xor_base_2 = BENCHMARK(2,2)
    for epoch in range(2000):
        for k in rng.permutation(4):
            xor_test.learn(P[k],T[k],1.0) 
            xor_test_2.learn(P[k],T[k],1.0) 
            xor_base.learn(P[k],T[k],0.05)
            xor_base_2.learn(P[k],T[k],0.05)
    forward_result = []
    for count in range(4):
        forward_result.append(xor_test.forward(P[count]))
    score = output_correct(forward_result)
    if score == 4:
        r1 += 1
    forward_result = []
    for count in range(4):
        forward_result.append(xor_test_2.forward(P[count]))
    score = output_correct(forward_result)
    if score == 4:
        r1_2 += 1
    forward_result = []
    for count in range(4):
        forward_result.append(xor_base.forward(P[count]))
    score = output_correct(forward_result)
    if score == 4:
        r2 += 1
    forward_result = []
    for count in range(4):
        forward_result.append(xor_base_2.forward(P[count]))
    score = output_correct(forward_result)
    if score == 4:
        r2_2 += 1
    return r1,r1_2,r2,r2_2

In [7]:
#SETUP OUTPUT VARIABLES
test_score,base_score,test_score_2,base_score_2 = 0,0,0,0
test_a_val,test_b_val,test_aa_val = 0,0,0
test_a_val_2,test_b_val_2,test_aa_val_2 = 0,0,0
base_a_val,base_b_val,base_aa_val = 0,0,0
base_a_val_2,base_b_val_2,base_aa_val_2 = 0,0,0

#RUN THE MODELS
seed_samples = 100
for i in range(seed_samples):
    print(i,end="|")
    
    #XOR TESTS
    returned_score = test_seed(i)
    test_score += returned_score[0]
    test_score_2 += returned_score[1]
    base_score += returned_score[2]
    base_score_2 += returned_score[3]
    
    #TREE TESTS ON A/B
    returned_test_val = memory_test(i,True,None)
    test_a_val += returned_test_val[0]
    test_b_val += returned_test_val[1]
    test_aa_val +=  returned_test_val[2]
    returned_test_val = memory_test(i,False,None)
    test_a_val_2 += returned_test_val[0]
    test_b_val_2 += returned_test_val[1]
    test_aa_val_2 +=  returned_test_val[2]
    
    #BASE TESTS ON A/B
    returned_base_val = memory_base(i,4)
    base_a_val += returned_base_val[0]
    base_b_val += returned_base_val[1]
    base_aa_val +=  returned_base_val[2]
    returned_base_val = memory_base(i,2)
    base_a_val_2 += returned_base_val[0]
    base_b_val_2 += returned_base_val[1]
    base_aa_val_2 +=  returned_base_val[2]

#PRINT A/B
print("\n\nMEMORY TEST OF TREE(Grow): \n-A:",test_a_val / seed_samples,"\n-B:",test_b_val / seed_samples,"\n-AA:",test_aa_val / seed_samples)
print("\nMEMORY TEST OF TREE(static): \n-A:",test_a_val_2 / seed_samples,"\n-B:",test_b_val_2 / seed_samples,"\n-AA:",test_aa_val_2 / seed_samples)
print("\nMEMORY TEST OF BASE(h=4): \n-A:",base_a_val / seed_samples,"\n-B:",base_b_val / seed_samples,"\n-AA:",base_aa_val / seed_samples)
print("\nMEMORY TEST OF BASE(=2): \n-A:",base_a_val_2 / seed_samples,"\n-B:",base_b_val_2 / seed_samples,"\n-AA:",base_aa_val_2 / seed_samples)

#PRINT XOR
print("\nXOR TEST OF TREE(Grow):",test_score)
print("XOR TEST OF TREE(static):",test_score_2)
print("XOR TEST OF BASE(h=4):",base_score)
print("XOR TEST OF BASE(h=2):",base_score_2)

0|1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|19|20|21|22|23|24|25|26|27|28|29|30|31|32|33|34|35|36|37|38|39|40|41|42|43|44|45|46|47|48|49|50|51|52|53|54|55|56|57|58|59|60|61|62|63|64|65|66|67|68|69|70|71|72|73|74|75|76|77|78|79|80|81|82|83|84|85|86|87|88|89|90|91|92|93|94|95|96|97|98|99|

MEMORY TEST OF TREE(Grow): 
-A: 0.0412031089619453 
-B: 0.014287439308225727 
-AA: 0.12396797132300053

MEMORY TEST OF TREE(static): 
-A: 0.06429927028687453 
-B: 0.05892421698539018 
-AA: 0.5784281721208625

MEMORY TEST OF BASE(h=4): 
-A: 0.00029311037210327356 
-B: 0.0003412726755191765 
-AA: 0.2905957274940853

MEMORY TEST OF BASE(=2): 
-A: 0.00017093443936280406 
-B: 0.0001753600335934896 
-AA: 0.2964766769240718

XOR TEST OF TREE(Grow): 100
XOR TEST OF TREE(static): 98
XOR TEST OF BASE(h=4): 99
XOR TEST OF BASE(h=2): 79


In [8]:
pair_rng = np.random.default_rng(123)
pairs,ratios = [],[]
while len(pairs) < 10:
    pair = pair_rng.uniform(-0.5,0.5,2)
    if abs(pair[0] - pair[1]) > 0.15:
        pairs.append(pair)
print(pairs)
for a,b in pairs:
    B,AA = [0,0,0,0],[0,0,0,0]
    YA = np.full(500,a)
    YB = np.full(500,b)
    for seed in range(5):
        xa = pair_rng.uniform(0.1,0.5,500)
        xb = pair_rng.uniform(-0.5,-0.1,500)
        XA = xa[:,None]
        XB = xb[:,None]
        re = memory_test(seed,True,None)
        B[0]  += re[1]
        AA[0] += re[2]
        re = memory_test(seed,False,None)
        B[1]  += re[1]
        AA[1] += re[2]
        re = memory_base(seed,4)
        B[2]  += re[1]
        AA[2] += re[2]
        re = memory_base(seed,2)
        B[3]  += re[1]
        AA[3] += re[2]
    test_both = (B[0] + AA[0]) / (2 * 5)
    base_both = (B[2] + AA[2]) / (2 * 5)
    ratio = base_both / test_both
    ratios.append(ratio)
    print(a,b,"|",ratio)


[array([ 0.18235186, -0.44617898]), array([-0.3240941 ,  0.31209451]), array([ 0.423345 , -0.2234256]), array([ 0.01297046, -0.2550354 ]), array([ 0.3242416 , -0.28623704]), array([ 0.42740726, -0.26809181]), array([0.29912513, 0.01816504]), array([-0.31566201, -0.48510508]), array([-0.02886677,  0.22824333]), array([0.41860049, 0.12553401])]
0.18235186324814345 -0.4461789811977773 | 10.92848095596509
-0.32409409891496965 0.31209450665577365 | 9.821472835996342
0.4233449980270564 -0.22342560220289376 | 11.271330471962514
0.012970455229531885 -0.2550353989312035 | 1.0478646682065698
0.3242415960974113 -0.2862370366249045 | 8.929254978657301
0.42740725852516703 -0.2680918113935812 | 11.347257821679866
0.2991251286200829 0.0181650368527142 | 2.1071844850210675
-0.31566201257152027 -0.48510508323976775 | 1.3628903326837853
-0.02886677110953917 0.22824332818326165 | 0.7370490793645061
0.4186004917735433 0.12553400573546403 | 2.3954097690800418
